<a href="https://colab.research.google.com/github/SumitJainUTD/AI-projects/blob/main/lunarlanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Learning en el entorno LunarLander-v3

En primer lugar instalamos las dependencias necesarias para ejecutar el código. En este caso, necesitamos instalar la biblioteca `gymnasium` y su extensión `box2d` para poder utilizar el entorno LunarLander-v3:

In [1]:
!pip install gymnasium
!pip install gymnasium[box2d]


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Realizamos los import necesarios, destacando en este caso la inclusión de `torch` para la implementación de la red neuronal que utilizaremos como función aproximadora en nuestro agente de Deep Q-Learning:

In [2]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
from agents import LunarAgentDeepQLearning

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)


Creamos la clase para la red neuronal que utilizaremos como función aproximadora en nuestro agente de Deep Q-Learning. Esta red tendrá una arquitectura simple con dos capas ocultas y una capa de salida que corresponde a las acciones posibles en el entorno:

También necesitaremos una clase para gestionar la memoria de experiencia, que nos permitirá almacenar las transiciones y muestrear mini-batches para el entrenamiento de la red neuronal:

In [3]:
def train_agent(env, agent, n_episodes=2000, max_t=1000, eps_start=1.0, eps_end=0.01, eps_decay=0.995):
    scores = []                        # Lista con todas las puntuaciones
    scores_window = deque(maxlen=100)  # Cola para calcular la media movil de los ultimos 100
    eps = eps_start                    # Inicializamos epsilon

    for i_episode in range(1, n_episodes + 1):
        state, _ = env.reset(seed=seed)
        score = 0
        
        # Iteramos los pasos del episodio
        for t in range(max_t):
            # Solicitamos la accion al agente
            action = agent.get_action(state, eps)
            next_state, reward, done, _, _ = env.step(action)
            
            # El agente procesa la transicion y aprende si es el momento
            agent.update(state, action, reward, done, next_state)
            
            state = next_state
            score += reward
            
            if done:
                break 
                
        # Guardamos las puntuaciones y reducimos epsilon
        scores_window.append(score)
        scores.append(score)
        agent.decay_epsilon()

        # Mostramos el progreso dinamicamente
        print(f'\rEpisodio {i_episode}\tMedia ultimos 100: {np.mean(scores_window):.2f}', end="")
        if i_episode % 100 == 0:
            print(f'\rEpisodio {i_episode}\tMedia ultimos 100: {np.mean(scores_window):.2f}')
            
    
    torch.save(agent.qnetwork_local.state_dict(), 'checkpoint_lunar_lander.pth')
            
            
    return scores

In [4]:
import gymnasium as gym
env = gym.make('LunarLander-v3') 
state_shape = env.observation_space.shape
state_size = env.observation_space.shape[0]
number_actions = env.action_space.n
print('State shape: ', state_shape)
print('State size: ', state_size)
print('Number of actions: ', number_actions)

State shape:  (8,)
State size:  8
Number of actions:  4


In [ ]:
number_episodes = 2000
maximum_number_timesteps_per_episode = 1000
epsilon_starting_value  = 1.0
epsilon_ending_value  = 0.01
ratio_decay = 0.5
discount_factor = 0.99
epsilon_decay_value  = epsilon_starting_value/number_episodes*ratio_decay
epsilon = epsilon_starting_value
scores_on_100_episodes = deque(maxlen = 100)


agent = LunarAgentDeepQLearning(env, 
                                state_size, 
                                number_actions, 
                                epsilon_starting_value, 
                                0.985, 
                                epsilon_ending_value, 
                                discount_factor, 
                                seed)
scores = train_agent(env, 
                     agent, 
                     n_episodes=number_episodes, 
                     max_t=maximum_number_timesteps_per_episode, 
                     eps_start=epsilon_starting_value, 
                     eps_end=epsilon_ending_value,
                     eps_decay=epsilon_decay_value
                     )

Episodio 100	Media ultimos 100: -207.95
Episodio 200	Media ultimos 100: -195.06
Episodio 300	Media ultimos 100: -228.22
Episodio 400	Media ultimos 100: -195.14
Episodio 500	Media ultimos 100: -209.29
Episodio 600	Media ultimos 100: -206.11
Episodio 700	Media ultimos 100: -204.19
Episodio 800	Media ultimos 100: -197.61
Episodio 900	Media ultimos 100: -215.22
Episodio 1000	Media ultimos 100: -190.45
Episodio 1100	Media ultimos 100: -222.79
Episodio 1200	Media ultimos 100: -224.44
Episodio 1300	Media ultimos 100: -205.98
Episodio 1400	Media ultimos 100: -200.83
Episodio 1500	Media ultimos 100: -226.83
Episodio 1600	Media ultimos 100: -204.22
Episodio 1700	Media ultimos 100: -218.70
Episodio 1800	Media ultimos 100: -221.71
Episodio 1900	Media ultimos 100: -217.58
Episodio 2000	Media ultimos 100: -207.79


In [6]:
agent.qnetwork_local.eval()
agent.test(env, n_episodes=100)

Episodio 1: Recompensa = -83.64
Episodio 2: Recompensa = -53.91
Episodio 3: Recompensa = -40.77
Episodio 4: Recompensa = -47.40
Episodio 5: Recompensa = -26.19
Episodio 6: Recompensa = -58.35
Episodio 7: Recompensa = -23.86
Episodio 8: Recompensa = -59.72
Episodio 9: Recompensa = -23.14
Episodio 10: Recompensa = -103.05
Episodio 11: Recompensa = -67.57
Episodio 12: Recompensa = -71.86
Episodio 13: Recompensa = -75.63
Episodio 14: Recompensa = -102.24
Episodio 15: Recompensa = -22.74
Episodio 16: Recompensa = -44.04
Episodio 17: Recompensa = -72.63
Episodio 18: Recompensa = -62.79
Episodio 19: Recompensa = -67.03
Episodio 20: Recompensa = -21.02
Episodio 21: Recompensa = -2.82
Episodio 22: Recompensa = -66.24
Episodio 23: Recompensa = -85.41
Episodio 24: Recompensa = -90.74
Episodio 25: Recompensa = -111.62
Episodio 26: Recompensa = -22.47
Episodio 27: Recompensa = -85.81
Episodio 28: Recompensa = -33.51
Episodio 29: Recompensa = -71.43
Episodio 30: Recompensa = -54.62
Episodio 31: Reco

In [ ]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env_name):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset()
    done = False
    frames = []
    while not done:
        frame = env.render()
        frames.append(frame)
        action = agent.get_action(state, eps=0.0)  # Sin exploración durante la prueba
        state, reward, done, _, _ = env.step(action.item())
    env.close()
    imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent, 'LunarLander-v3')

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()